> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board and uploads the
> package. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host,
>   writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

# Hardware and configuration

| | Host run | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | none | none: `RoutingTable` only reads a config, it never touches the bus |
| **Driver registry** | stand-in classes with the real parts' `ALLOWED_ADDRS` and `CHANNELS`; one integration test uses the real `devices.registry()` | the real `devices.registry()` for the bench's six drivers |
| **Result** | not run yet on the hardware | not run yet |

# TODO:
Nothing open.

Fixed (2026-11-16), found while porting:
- `ConfigError` lost its `problems` in a pickle round trip; it now pickles by its problems.
- A missing required key raised a bare `KeyError` from `from_dict`, and an address that does
  not parse a `ValueError`; both are now a `ConfigError` listing every such entry.
- A channel that is not hashable (`[3]`) crashed `segments()` with `TypeError`, and `True`
  passed as channel 1. A channel must be a plain non-negative `int`; with a bad channel the
  collision and channel-range checks are skipped (they need valid channels).
- A mux whose driver has no `CHANNELS` (a device driver used as a mux) raised
  `AttributeError`; it is now reported as "not a mux driver".

Decided (2026-11-16): the unit tests use stand-in driver classes (only `ALLOWED_ADDRS`,
`CHANNELS` and `addr_ok`), so a broken driver cannot break the routing tests. One
integration test and the hardware section use the real registry.

# Python file
location: `src/micropython_i2c/routing.py`

tests: `tests/test_routing.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/routing.py
"""``RoutingTable``: the declarative bus topology, ingested and validated.

Config format (JSON text or an equivalent dict)::

    {
      "buses": [0],
      "muxes": {
        "mux_a": {"bus": 0, "address": "0x71", "path": [], "driver": "TCA9548A",
                  "reset_pin": 5}
      },
      "devices": [
        {"name": "hall", "bus": 0, "address": "0x35", "path": [["mux_a", 3]],
         "driver": "TMAG5273", "init_kwargs": {}}
      ]
    }

A ``path`` is the full route, outermost hop first, naming muxes by their config name.
A mux's own ``path`` is the route that reaches *it*. Addresses may be ints or "0x.."
strings.

``validate()`` collects every problem into one ``ConfigError`` rather than stopping at
the first. Structural checks always run; driver-aware checks (driver exists, address
legal for the part, channel within the mux's ``CHANNELS``) run when a registry
``{driver name: class}`` is passed.

Generated by ``dev_notebooks/routing_devNB``: edit the notebook, not this file.
"""

# Imports
## External
`json`, for config text. MicroPython has it under the same name.

## Internal
None: the driver registry is passed in by the caller, so this module imports no driver.

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
import json

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_routing.py
import pickle

import pytest

from micropython_i2c.devices import registry
from micropython_i2c.routing import ConfigError, RoutingTable

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pickle

import pytest

from micropython_i2c.devices import registry

# Intent
**Purpose:** turn the bench's wiring, written once as a config, into a checked table: which
buses exist, where each mux sits, and the full route (bus, mux hops, address) to every
device. The manager builds live objects from it; `scan_check` probes it.

**Why it exists:** a wrong config shows up on the board as a NACK, a part answering for
another, or a mux left open. Every one of those mistakes can be seen in the config itself, so
`validate()` finds them all at once, before anything touches the bus.

**What `validate()` checks, in order:**

```mermaid
flowchart TD
    A[mux references exist] --> B[buses declared]
    B --> C[names unique]
    C --> D[channels are ints >= 0]
    D --> E{references OK?}
    E -->|yes| F[path consistency]
    F -->|consistent| G[address collisions]
    D --> H{registry given?}
    H -->|yes| I[driver exists, address legal]
    I --> J[channel < mux CHANNELS, if references OK]
```

Checks that need a sound topology are skipped when an earlier check found it unsound, so one
typo does not produce a cascade of follow-on problems.

**Segments and collisions.** A segment is `(bus, path)`: the parts that answer when exactly
that path is selected. Selecting a path leaves every upstream segment connected too, so the
parts visible on a segment are its own plus those of every prefix of its path. Two entries
collide when they share an address and are visible together. Parts on sibling channels of
one mux never are.

| Path selected | Visible segments |
|---|---|
| `[]` | `[]` |
| `[("mux_a", 3)]` | `[]`, `[("mux_a", 3)]` |
| `[("mux_a", 3), ("mux_b", 0)]` | `[]`, `[("mux_a", 3)]`, `[("mux_a", 3), ("mux_b", 0)]` |

**Used by:** `manager.I2CManager` and `manager.scan_check`, the bench config in `sim.bench`,
`notebooks/03_routing_harness`.

**Not its job:** talking to the bus, importing drivers (the caller passes the registry), or
checking bus speeds (the manager does that).

## What the module defines

| Name | What it is |
|---|---|
| `ConfigError` | the error `validate()` raises, with every problem in `problems` |
| `RoutingTable` | the parsed topology: `loads`, `from_dict`, the views `entries`, `resolve_path`, `segments`, and `validate` |

# Code
## TODO:
Nothing

## Errors and warnings
One error, raised by `validate()` and, for entries it cannot parse, by `from_dict()`:

```
Exception
└── ConfigError
```

| Error | Also a | Raised when |
|---|---|---|
| `ConfigError` | `Exception` | `validate()` found one or more problems, or `from_dict()` met an entry missing a required key or with an address that does not parse |

Example message:

```
routing table invalid:
  hall: path references unknown mux 'nope'
  hall: bus 0 not in declared buses [1]
```

A missing required key (`name`, `bus`, `address`, `driver` on a device; `bus`, `address` on
a mux) is reported by `from_dict`, since the table cannot be built without it:

```
routing table invalid:
  devices[0]: missing required key 'driver'
  mux:mux_a: address 'zz' does not parse
```

It pickles by its problems, so it survives a round trip (a worker process, a test runner).

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
class ConfigError(Exception):
    """A routing table failed validation. ``problems`` lists every issue found."""

    def __init__(self, problems):
        self.problems = list(problems)
        super().__init__("routing table invalid:\n  " + "\n  ".join(self.problems))

    def __reduce__(self):
        return (self.__class__, (self.problems,))

### validation/test

#### validation: nominal usage
Build one from two problems and print it: the message lists each on its own line.

In [ ]:
err = ConfigError(["hall: path references unknown mux 'nope'", "hall: bus 0 not in declared buses [1]"])
print("problems:", err.problems)
print(err)

#### validation: anti-nominal usage
`ConfigError` is a plain `Exception`, so a generic handler still catches it.

In [ ]:
try:
    raise ConfigError(["x: no driver given"])  # an invalid table
except Exception as err:  # caught as its base
    print("raised ConfigError, caught as Exception, as expected:\n    {}".format(err.problems))
else:
    raise AssertionError("raise should have raised")

#### unit

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_config_error_lists_every_problem():
    err = ConfigError(("a: first", "b: second"))
    assert err.problems == ["a: first", "b: second"]
    assert str(err) == "routing table invalid:\n  a: first\n  b: second"
    assert isinstance(err, Exception)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_config_error_survives_a_pickle_round_trip():
    err = pickle.loads(pickle.dumps(ConfigError(["a: first", "b: second"])))
    assert err.problems == ["a: first", "b: second"]
    assert str(err) == "routing table invalid:\n  a: first\n  b: second"

## Public names

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
__all__ = [
    "RoutingTable",
    # errors
    "ConfigError",
]

## Parsing helpers
`_parse_addr` accepts an address as an int, a `"0x.."` string, or a decimal string, so a
config can use whatever the datasheet uses. JSON has no hex literals, hence the strings.

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
def _parse_addr(value):
    if isinstance(value, str):
        return int(value, 16) if value.lower().startswith("0x") else int(value)
    return value

`_parse_path` turns the JSON hops (`[["mux_a", 3]]`, lists because JSON has no tuples) into
`(name, channel)` tuples, so a path can be a dict key in `segments()`. A missing or `null`
path is the root segment, `[]`.

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
def _parse_path(path):
    return [(hop[0], hop[1]) for hop in path or ()]

`_missing` and `_parse_or_report` let `from_dict` collect every entry it cannot parse into one
`ConfigError`, instead of stopping at the first `KeyError` or `ValueError`.

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
def _missing(problems, label, entry, keys):
    """Report each of ``keys`` absent from ``entry``; True if any was."""
    absent = [k for k in keys if k not in entry]
    for k in absent:
        problems.append("{}: missing required key {!r}".format(label, k))
    return bool(absent)


def _parse_or_report(problems, label, what, parse, raw):
    """``parse(raw)``, or report that it does not parse and return ``raw``."""
    try:
        return parse(raw)
    except (ValueError, TypeError, IndexError, KeyError):
        problems.append("{}: {} {!r} does not parse".format(label, what, raw))
        return raw

### validation/test
Validated through `RoutingTable.loads` below (`test_addresses_parse_from_hex_decimal_and_int`,
`test_paths_become_tuples`): the helpers are private.

## `RoutingTable`
The parsed topology and its checks. Build it with `loads()` (JSON text or a dict) or
`from_dict()`, then call `validate()`, which returns the table so the two chain:
`RoutingTable.loads(text).validate(registry())`.

Each mux becomes `{"bus", "address", "path", "reset_pin", "driver"}`, each device
`{"name", "bus", "address", "path", "driver", "init_kwargs"}`. The views (`entries`,
`resolve_path`, `segments`) are what the checks, the scanner and the manager read. Every
check appends to one `problems` list; `validate()` raises once, with all of them.

In [ ]:
%%extract ../src/micropython_i2c/routing.py -a
class RoutingTable:
    """Parsed topology. Build with ``loads()`` or ``from_dict()``, then ``validate()``.

    Attributes:
        buses: Declared bus ids.
        muxes: ``{name: {"bus", "address", "path", "reset_pin", "driver"}}``.
        devices: List of ``{"name", "bus", "address", "path", "driver", "init_kwargs"}``.
    """

    def __init__(self, buses, muxes, devices):
        self.buses = buses
        self.muxes = muxes
        self.devices = devices

    @classmethod
    def loads(cls, raw):
        """Build from JSON text, or from a dict."""
        return cls.from_dict(raw if isinstance(raw, dict) else json.loads(raw))

    @classmethod
    def from_dict(cls, data):
        """Build from a parsed config dict.

        Raises:
            ConfigError: If an entry lacks a required key (``bus``, ``address`` on a mux; also
                ``name`` and ``driver`` on a device) or has an address or path that does not
                parse. Every such entry is listed.
        """
        problems = []
        muxes = {}
        for name, m in data.get("muxes", {}).items():
            label = "mux:{}".format(name)
            if _missing(problems, label, m, ("bus", "address")):
                continue
            muxes[name] = {
                "bus": m["bus"],
                "address": _parse_or_report(problems, label, "address", _parse_addr, m["address"]),
                "path": _parse_or_report(problems, label, "path", _parse_path, m.get("path")),
                "reset_pin": m.get("reset_pin"),
                "driver": m.get("driver"),
            }
        devices = []
        for i, d in enumerate(data.get("devices", [])):
            label = d.get("name", "devices[{}]".format(i))
            if _missing(problems, label, d, ("name", "bus", "address", "driver")):
                continue
            devices.append(
                {
                    "name": d["name"],
                    "bus": d["bus"],
                    "address": _parse_or_report(
                        problems, label, "address", _parse_addr, d["address"]
                    ),
                    "path": _parse_or_report(problems, label, "path", _parse_path, d.get("path")),
                    "driver": d["driver"],
                    "init_kwargs": d.get("init_kwargs", {}),
                }
            )
        if problems:
            raise ConfigError(problems)
        return cls(list(data.get("buses", [])), muxes, devices)

    # Views used by the checks, the scanner and the manager.

    def entries(self):
        """Yield ``(label, entry)`` for every mux (labelled ``mux:<name>``) and device."""
        for name, m in self.muxes.items():
            yield "mux:" + name, m
        for d in self.devices:
            yield d["name"], d

    def resolve_path(self, path):
        """Turn ``[(mux_name, ch), ...]`` into ``[[mux_addr, ch], ...]``."""
        return [[self.muxes[name]["address"], ch] for name, ch in path]

    def segments(self):
        """``{(bus, path_tuple): [(address, label), ...]}`` for every entry."""
        segs = {}
        for label, e in self.entries():
            segs.setdefault((e["bus"], tuple(e["path"])), []).append((e["address"], label))
        return segs

    # Structural checks. Each appends to ``problems``.

    def _check_mux_refs(self, problems):
        ok = True
        for label, e in self.entries():
            for name, _ in e["path"]:
                if name not in self.muxes:
                    problems.append("{}: path references unknown mux {!r}".format(label, name))
                    ok = False
        return ok

    def _check_buses_declared(self, problems):
        for label, e in self.entries():
            if e["bus"] not in self.buses:
                problems.append(
                    "{}: bus {} not in declared buses {}".format(label, e["bus"], self.buses)
                )

    def _check_names_unique(self, problems):
        counts = {}
        for d in self.devices:
            counts[d["name"]] = counts.get(d["name"], 0) + 1
        for name, n in counts.items():
            if n > 1:
                problems.append("duplicate device name {!r} ({} entries)".format(name, n))
            if name in self.muxes:
                problems.append("name {!r} used by both a device and a mux".format(name))

    def _check_channels_valid(self, problems):
        """Every channel is a plain non-negative int (not a bool). True if all are."""
        ok = True
        for label, e in self.entries():
            for name, ch in e["path"]:
                if isinstance(ch, bool) or not isinstance(ch, int) or ch < 0:
                    problems.append("{}: bad channel {!r} on mux {!r}".format(label, ch, name))
                    ok = False
        return ok

    def _check_path_consistency(self, problems):
        """Every hop's mux must sit on the same bus, reached by exactly the preceding hops.

        This also rules out cycles and a mux appearing twice in one path, since a mux
        cannot be reached by two different prefixes.
        """
        ok = True
        for label, e in self.entries():
            for i, (name, _) in enumerate(e["path"]):
                m = self.muxes[name]
                if m["bus"] != e["bus"]:
                    problems.append(
                        "{}: on bus {} but its path enters mux {!r} on bus {}".format(
                            label, e["bus"], name, m["bus"]
                        )
                    )
                    ok = False
                if m["path"] != e["path"][:i]:
                    problems.append(
                        "{}: hop {} goes through mux {!r} via {} but that mux is "
                        "declared at {}".format(label, i, name, e["path"][:i], m["path"])
                    )
                    ok = False
        return ok

    def _check_addr_collisions(self, problems):
        """Two entries collide if both are visible when one segment is selected.

        Selecting a path leaves every upstream segment connected, so a segment sees its
        own members plus those of every prefix of its path.
        """
        segs = self.segments()
        reported = set()
        for (bus, path), _ in segs.items():
            by_addr = {}
            for i in range(len(path) + 1):
                for addr, label in segs.get((bus, path[:i]), ()):
                    by_addr.setdefault(addr, []).append(label)
            for addr, labels in by_addr.items():
                key = (bus, addr, tuple(sorted(labels)))
                if len(labels) > 1 and key not in reported:
                    reported.add(key)
                    problems.append(
                        "address collision on bus {} at {:#04x}: {}".format(
                            bus, addr, ", ".join(labels)
                        )
                    )

    # Driver-aware checks.

    def _check_drivers(self, problems, registry):
        for label, e in self.entries():
            drv = e.get("driver")
            if drv is None:
                problems.append("{}: no driver given".format(label))
                continue
            cls = registry.get(drv)
            if cls is None:
                problems.append("{}: driver {!r} not in registry".format(label, drv))
                continue
            if label.startswith("mux:") and getattr(cls, "CHANNELS", None) is None:
                problems.append(
                    "{}: driver {!r} is not a mux driver (no CHANNELS)".format(label, drv)
                )
            if not cls.addr_ok(e["address"]):
                problems.append(
                    "{}: address {:#04x} not valid for {} (allowed {})".format(
                        label, e["address"], drv, [hex(a) for a in cls.ALLOWED_ADDRS]
                    )
                )

    def _check_channels_vs_class(self, problems, registry):
        for label, e in self.entries():
            for name, ch in e["path"]:
                channels = getattr(registry.get(self.muxes[name].get("driver")), "CHANNELS", None)
                if channels is not None and not ch < channels:
                    problems.append(
                        "{}: channel {} out of range 0..{} for mux {!r} ({})".format(
                            label, ch, channels - 1, name, self.muxes[name]["driver"]
                        )
                    )

    def validate(self, registry=None):
        """Run every check; raise ``ConfigError`` listing all problems. Returns ``self``."""
        problems = []
        refs_ok = self._check_mux_refs(problems)
        self._check_buses_declared(problems)
        self._check_names_unique(problems)
        channels_ok = self._check_channels_valid(problems)
        if refs_ok and channels_ok and self._check_path_consistency(problems):
            self._check_addr_collisions(problems)
        if registry is not None:
            self._check_drivers(problems, registry)
            if refs_ok and channels_ok:
                self._check_channels_vs_class(problems, registry)
        if problems:
            raise ConfigError(problems)
        return self

### validation/test: fixtures
The registry, config and helper every check below uses. They are written to the test file
too.

Stand-in driver classes: the routing checks read only `ALLOWED_ADDRS`, `CHANNELS` and
`addr_ok` from a driver class, so these carry just that, copied from the real drivers. A
broken driver module then cannot break the routing tests. Their `__name__` is the real
class name, because the channel message prints it.

In [ ]:
%%extract ../tests/test_routing.py -a
class StandInPart:
    """A driver class reduced to what ``RoutingTable`` reads."""

    ALLOWED_ADDRS = ()

    @classmethod
    def addr_ok(cls, addr):
        return addr in cls.ALLOWED_ADDRS


def stand_in(name, addrs, channels=None):
    """A ``StandInPart`` subclass called ``name``; a mux when ``channels`` is given."""
    attrs = {"ALLOWED_ADDRS": tuple(addrs)}
    if channels is not None:
        attrs["CHANNELS"] = channels
    return type(name, (StandInPart,), attrs)

The stand-in registry: the six drivers the bench uses, with the real parts' addresses (from
each driver's `ALLOWED_ADDRS`).

In [ ]:
%%extract ../tests/test_routing.py -a
REG = {
    "SHTC3": stand_in("SHTC3", [0x70]),
    "TMAG5273": stand_in("TMAG5273", range(0x08, 0x79)),
    "TCA9548A": stand_in("TCA9548A", range(0x70, 0x78), channels=8),
    "TMP102": stand_in("TMP102", [0x48, 0x49, 0x4A, 0x4B]),
    "VL53L4CD": stand_in("VL53L4CD", range(0x08, 0x78)),
    "QwiicRelay": stand_in("QwiicRelay", [0x18, 0x19]),
}

A small valid config, made fresh on every call so a check can change it: a mux at `0x71`, an
SHTC3 on the root segment, a TMAG5273 behind mux channel 3.

In [ ]:
%%extract ../tests/test_routing.py -a
def config(**overrides):
    cfg = {
        "buses": [0],
        "muxes": {"mux_a": {"bus": 0, "address": "0x71", "driver": "TCA9548A"}},
        "devices": [
            {"name": "humidity", "bus": 0, "address": "0x70", "driver": "SHTC3"},
            {"name": "hall", "bus": 0, "address": "0x35", "driver": "TMAG5273",
             "path": [["mux_a", 3]]},
        ],
    }  # fmt: skip
    cfg.update(overrides)
    return cfg

`problems(cfg)`: validate a config that must fail, and return its problems as one text, for
substring checks.

In [ ]:
%%extract ../tests/test_routing.py -a
def problems(cfg, registry=REG):
    with pytest.raises(ConfigError) as info:
        RoutingTable.loads(cfg).validate(registry)
    return "\n".join(info.value.problems)

The bench as wired: the mux on bus 0 with the SHTC3 (channel 0) and TMP102 (channel 2) behind
it, the VL53L4CD and relay on bus 1.

In [ ]:
%%extract ../tests/test_routing.py -a
BENCH = {
    "buses": [0, 1],
    "muxes": {"mux": {"bus": 0, "address": "0x71", "driver": "TCA9548A"}},
    "devices": [
        {"name": "humidity", "bus": 0, "address": "0x70", "driver": "SHTC3",
         "path": [["mux", 0]]},
        {"name": "temp", "bus": 0, "address": "0x48", "driver": "TMP102",
         "path": [["mux", 2]]},
        {"name": "tof", "bus": 1, "address": "0x29", "driver": "VL53L4CD"},
        {"name": "relay", "bus": 1, "address": "0x18", "driver": "QwiicRelay"},
    ],
}  # fmt: skip

### validation/test: parsing and views

#### validation: nominal usage
Parse the small config and print what the manager and scanner read: the parsed entries, the
resolved route to the hall sensor, and the segments.

In [ ]:
table = RoutingTable.loads(config())
print("buses:", table.buses)
print("mux_a:", table.muxes["mux_a"])
for label, e in table.entries():
    print("entry {:<10} bus {} addr {:#04x} path {}".format(label, e["bus"], e["address"], e["path"]))
print("route to hall:", table.resolve_path(table.devices[1]["path"]))
print("segments:", table.segments())

#### validation: anti-nominal usage
A device without a `driver` key, and a mux whose address does not parse: `from_dict` reports
both in one `ConfigError` before any check runs.

In [ ]:
cfg = config()
del cfg["devices"][0]["driver"]
cfg["muxes"]["mux_a"]["address"] = "zz"
try:
    RoutingTable.loads(cfg)  # a device needs a driver; an address must parse
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a device without a driver should have raised ConfigError")

#### unit

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_valid_config_passes_and_parses_hex():
    table = RoutingTable.loads(config()).validate(REG)
    assert table.muxes["mux_a"]["address"] == 0x71
    assert table.resolve_path(table.devices[1]["path"]) == [[0x71, 3]]

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_json_text_accepted():
    RoutingTable.loads('{"buses": [0], "devices": []}').validate()

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_addresses_parse_from_hex_decimal_and_int():
    cfg = config()
    cfg["devices"][0]["address"] = "112"
    cfg["devices"][1]["address"] = 0x35
    cfg["muxes"]["mux_a"]["address"] = "0X71"
    table = RoutingTable.loads(cfg)
    assert [e["address"] for _, e in table.entries()] == [0x71, 0x70, 0x35]

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_paths_become_tuples_and_defaults_fill_in():
    table = RoutingTable.loads(config())
    assert table.devices[1]["path"] == [("mux_a", 3)]
    assert table.devices[0]["path"] == []
    assert table.devices[0]["init_kwargs"] == {}
    assert table.muxes["mux_a"]["reset_pin"] is None
    assert table.muxes["mux_a"]["path"] == []

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_entries_label_muxes_and_segments_group_by_route():
    table = RoutingTable.loads(config())
    assert [label for label, _ in table.entries()] == ["mux:mux_a", "humidity", "hall"]
    assert table.segments() == {
        (0, ()): [(0x71, "mux:mux_a"), (0x70, "humidity")],
        (0, (("mux_a", 3),)): [(0x35, "hall")],
    }

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_missing_keys_and_bad_addresses_are_listed_together():
    cfg = config()
    del cfg["devices"][0]["driver"]
    del cfg["devices"][1]["name"]
    cfg["muxes"]["mux_a"]["address"] = "zz"
    with pytest.raises(ConfigError) as info:
        RoutingTable.loads(cfg)
    assert info.value.problems == [
        "mux:mux_a: address 'zz' does not parse",
        "humidity: missing required key 'driver'",
        "devices[1]: missing required key 'name'",
    ]

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_a_malformed_path_is_reported():
    cfg = config()
    cfg["devices"][1]["path"] = [["mux_a"]]  # a hop needs a mux and a channel
    with pytest.raises(ConfigError, match="hall: path .* does not parse"):
        RoutingTable.loads(cfg)

### validation/test: structural checks

#### validation: nominal usage
The small config passes every structural check; `validate()` returns the table itself.

In [ ]:
table = RoutingTable.loads(config())
print("validate() returns the table:", table.validate() is table)

#### validation: anti-nominal usage
One cell per mistake. The first shows that every problem is reported at once.

In [ ]:
cfg = config(buses=[1])
cfg["devices"][1]["path"] = [["nope", 0]]
try:
    RoutingTable.loads(cfg).validate()  # unknown mux, and bus 0 not declared
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("an unknown mux and an undeclared bus should have raised ConfigError")

In [ ]:
cfg = config()
cfg["devices"][1]["name"] = "humidity"
try:
    RoutingTable.loads(cfg).validate()  # two devices called "humidity"
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("a duplicate name should have raised ConfigError")

In [ ]:
cfg = config()
cfg["muxes"]["mux_b"] = {"bus": 0, "address": "0x72", "driver": "TCA9548A", "path": [["mux_a", 1]]}
cfg["devices"][1]["path"] = [["mux_b", 0]]
try:
    RoutingTable.loads(cfg).validate()  # mux_b is behind mux_a, but the path skips mux_a
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("a path skipping a hop should have raised ConfigError")

In [ ]:
cfg = config()
cfg["muxes"]["mux_a"]["path"] = [["mux_a", 0]]
try:
    RoutingTable.loads(cfg).validate()  # a mux reached through itself
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("a mux cycle should have raised ConfigError")

#### unit

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_unknown_mux_and_undeclared_bus():
    cfg = config(buses=[1])
    cfg["devices"][1]["path"] = [["nope", 0]]
    text = problems(cfg)
    assert "unknown mux 'nope'" in text and "not in declared buses" in text

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_duplicate_names():
    cfg = config()
    cfg["devices"][1]["name"] = "humidity"
    assert "duplicate device name" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_name_shared_by_a_device_and_a_mux():
    cfg = config()
    cfg["devices"][0]["name"] = "mux_a"
    assert "name 'mux_a' used by both a device and a mux" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_negative_or_non_int_channel_rejected():
    cfg = config()
    cfg["devices"][1]["path"] = [["mux_a", -1]]
    assert "bad channel -1 on mux 'mux_a'" in problems(cfg)
    cfg["devices"][1]["path"] = [["mux_a", "3"]]
    assert "bad channel '3' on mux 'mux_a'" in problems(cfg)
    cfg["devices"][1]["path"] = [["mux_a", True]]  # a bool is not a channel number
    assert "bad channel True on mux 'mux_a'" in problems(cfg)
    cfg["devices"][1]["path"] = [["mux_a", [3]]]  # unhashable: used to crash segments()
    assert "bad channel [3] on mux 'mux_a'" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_path_inconsistent_with_mux_declaration():
    cfg = config()
    cfg["muxes"]["mux_b"] = {"bus": 0, "address": "0x72", "driver": "TCA9548A",
                             "path": [["mux_a", 1]]}  # fmt: skip
    cfg["devices"][1]["path"] = [["mux_b", 0]]  # skips mux_a
    assert "declared at" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_path_through_a_mux_on_another_bus():
    cfg = config(buses=[0, 1])
    cfg["devices"][1]["bus"] = 1
    assert "hall: on bus 1 but its path enters mux 'mux_a' on bus 0" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_mux_cycle_rejected():
    cfg = config()
    cfg["muxes"]["mux_a"]["path"] = [["mux_a", 0]]
    assert "mux 'mux_a'" in problems(cfg)

### validation/test: address collisions

#### validation: nominal usage
Two TMAG5273s at the same address on sibling channels of one mux: never visible together, so
no collision.

In [ ]:
cfg = config()
cfg["devices"].append({"name": "hall2", "bus": 0, "address": "0x35", "driver": "TMAG5273", "path": [["mux_a", 4]]})
table = RoutingTable.loads(cfg).validate(REG)
for (bus, path), members in table.segments().items():
    print("bus {} path {}: {}".format(bus, list(path), members))

#### validation: anti-nominal usage

In [ ]:
cfg = config()
cfg["muxes"]["mux_a"]["address"] = "0x70"
try:
    RoutingTable.loads(cfg).validate()  # mux and SHTC3 both at 0x70 on the root segment
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("two parts at 0x70 on one segment should have raised ConfigError")

In [ ]:
cfg = config()
cfg["devices"][1]["address"] = "0x70"
try:
    RoutingTable.loads(cfg).validate()  # behind mux_a, but the root's 0x70 is still connected
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("a part shadowing an upstream address should have raised ConfigError")

#### unit

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_collision_on_same_segment():
    cfg = config()
    cfg["muxes"]["mux_a"]["address"] = "0x70"
    assert "collision" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_collision_seen_through_upstream_segment():
    cfg = config()
    cfg["devices"][1]["address"] = "0x70"  # behind mux_a, but humidity at 0x70 upstream
    cfg["devices"][1]["driver"] = "SHTC3"
    assert "humidity, hall" in problems(cfg) or "hall, humidity" in problems(cfg)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_same_address_on_sibling_channels_is_fine():
    cfg = config()
    cfg["devices"].append({"name": "hall2", "bus": 0, "address": "0x35", "driver": "TMAG5273",
                           "path": [["mux_a", 4]]})  # fmt: skip
    RoutingTable.loads(cfg).validate(REG)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_same_address_on_different_buses_is_fine():
    cfg = config(buses=[0, 1])
    cfg["devices"].append({"name": "humidity2", "bus": 1, "address": "0x70", "driver": "SHTC3"})
    RoutingTable.loads(cfg).validate(REG)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_each_collision_reported_once():
    cfg = config()
    cfg["muxes"]["mux_a"]["address"] = "0x70"
    found = problems(cfg).splitlines()
    assert found == ["address collision on bus 0 at 0x70: mux:mux_a, humidity"]

### validation/test: driver-aware checks

#### validation: nominal usage
With the registry, the small config still passes: every driver is known, every address is
legal for its part, and channel 3 is within the TCA9548A's eight.

In [ ]:
print("valid with the registry:", RoutingTable.loads(config()).validate(REG) is not None)

#### validation: anti-nominal usage
Three driver mistakes in one config, all reported: an SHTC3 at an address it cannot have, a
channel the TCA9548A does not have, and a driver nobody registered.

In [ ]:
cfg = config()
cfg["devices"][0]["address"] = "0x44"
cfg["devices"][1]["path"] = [["mux_a", 9]]
cfg["devices"].append({"name": "x", "bus": 0, "address": 1, "driver": "NOPE"})
try:
    RoutingTable.loads(cfg).validate(REG)  # wrong address, channel 9 of 8, unknown driver
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("three driver mistakes should have raised ConfigError")

#### unit

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_driver_aware_checks():
    cfg = config()
    cfg["devices"][0]["address"] = "0x44"
    cfg["devices"][1]["path"] = [["mux_a", 9]]
    cfg["devices"].append({"name": "x", "bus": 0, "address": 1, "driver": "NOPE"})
    text = problems(cfg)
    assert "not valid for SHTC3" in text
    assert "channel 9 out of range 0..7" in text
    assert "'NOPE' not in registry" in text

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_driver_checks_skipped_without_registry():
    cfg = config()
    cfg["devices"][0]["address"] = "0x44"
    RoutingTable.loads(cfg).validate()

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_mux_without_driver_reported_with_registry():
    cfg = config()
    del cfg["muxes"]["mux_a"]["driver"]
    assert "mux:mux_a: no driver given" in problems(cfg)
    RoutingTable.loads(cfg).validate()  # structural checks do not need a driver

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_a_device_driver_used_as_a_mux_is_reported():
    cfg = config()
    cfg["muxes"]["mux_a"]["driver"] = "TMAG5273"  # no CHANNELS: not a mux
    assert "mux:mux_a: driver 'TMAG5273' is not a mux driver (no CHANNELS)" in problems(cfg)

### validation/test: the bench

#### validation: nominal usage
The bench config validates, with the mux at `0x71`.

In [ ]:
bench = RoutingTable.loads(BENCH).validate(REG)
for label, e in bench.entries():
    print("{:<10} bus {} addr {:#04x} via {}".format(label, e["bus"], e["address"], bench.resolve_path(e["path"])))

#### validation: anti-nominal usage
The TCA9548A's default address is `0x70`, the SHTC3's fixed one. Left at the default, the
mux collides with the SHTC3 behind it: the reason the bench's mux is strapped to `0x71`.

In [ ]:
cfg = dict(BENCH)
cfg["muxes"] = {"mux": {"bus": 0, "address": "0x70", "driver": "TCA9548A"}}
try:
    RoutingTable.loads(cfg).validate(REG)  # mux at its default 0x70, SHTC3 behind it
except ConfigError as err:
    print("raised ConfigError, as expected:\n    {}".format("\n    ".join(err.problems)))
else:
    raise AssertionError("the mux at 0x70 should have raised ConfigError")

#### unit

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_bench_config_validates():
    RoutingTable.loads(BENCH).validate(REG)

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_bench_mux_at_default_address_collides_with_shtc3():
    cfg = {**BENCH, "muxes": {"mux": {"bus": 0, "address": "0x70", "driver": "TCA9548A"}}}
    assert "collision on bus 0 at 0x70" in problems(cfg)

#### integration
The same bench checks against the real driver classes, so a driver whose `ALLOWED_ADDRS` or
`CHANNELS` drifted from the stand-ins shows up here.

In [ ]:
%%extract ../tests/test_routing.py -a --strip-ipytest
%%ipytest


def test_bench_validates_with_the_real_drivers():
    real = registry(["SHTC3", "TMAG5273", "TCA9548A", "TMP102", "VL53L4CD", "QwiicRelay"])
    RoutingTable.loads(BENCH).validate(real)
    RoutingTable.loads(config()).validate(real)
    cfg = config()
    cfg["devices"][1]["path"] = [["mux_a", 8]]
    assert "channel 8 out of range 0..7" in problems(cfg, real)

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3 and uploads the
> package. Nothing here uses the I2C bus.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

`RoutingTable` is pure Python, so the board's job here is to show it behaves the same on
MicroPython: its `json` module, `int(text, 16)`, `{:#04x}` formatting, and the real driver
classes as imported on the board. Layout: connect and set up, the **matching tests**, then
the board and the host validate the same configs and compare the problems they find, then
the tally and disconnect.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use, load the real registry for the bench's six drivers, and zero the
tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

import json

from micropython_i2c.debug import check, check_raises, check_reset, check_summary
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.devices import registry
from micropython_i2c.routing import ConfigError, RoutingTable

DRIVER_NAMES = ["SHTC3", "TMAG5273", "TCA9548A", "TMP102", "VL53L4CD", "QwiicRelay"]
REG = registry(DRIVER_NAMES)
print("registry:", sorted(REG))

check_reset()

Notebook-only: the configs the checks below use, as JSON text, so the board's `json` parses
each one. `board_problems(name)` validates one with the real registry and returns its
problems, sorted (dict order may differ between MicroPython and CPython, and with it the
order problems are found in), or `[]` if it is valid.

**What to do:** nothing.

In [ ]:
SMALL = {
    "buses": [0],
    "muxes": {"mux_a": {"bus": 0, "address": "0x71", "driver": "TCA9548A"}},
    "devices": [
        {"name": "humidity", "bus": 0, "address": "0x70", "driver": "SHTC3"},
        {"name": "hall", "bus": 0, "address": "0x35", "driver": "TMAG5273", "path": [["mux_a", 3]]},
    ],
}
BENCH = {
    "buses": [0, 1],
    "muxes": {"mux": {"bus": 0, "address": "0x71", "driver": "TCA9548A"}},
    "devices": [
        {"name": "humidity", "bus": 0, "address": "0x70", "driver": "SHTC3", "path": [["mux", 0]]},
        {"name": "temp", "bus": 0, "address": "0x48", "driver": "TMP102", "path": [["mux", 2]]},
        {"name": "tof", "bus": 1, "address": "0x29", "driver": "VL53L4CD"},
        {"name": "relay", "bus": 1, "address": "0x18", "driver": "QwiicRelay"},
    ],
}


def variant(base, edit):
    """A deep copy of ``base`` (through JSON) with ``edit(cfg)`` applied, as JSON text."""
    cfg = json.loads(json.dumps(base))
    edit(cfg)
    return json.dumps(cfg)


def _unknown_mux_and_bus(cfg):
    cfg["buses"] = [1]
    cfg["devices"][1]["path"] = [["nope", 0]]


def _duplicate_name(cfg):
    cfg["devices"][1]["name"] = "humidity"


def _cycle(cfg):
    cfg["muxes"]["mux_a"]["path"] = [["mux_a", 0]]


def _upstream_collision(cfg):
    cfg["devices"][1]["address"] = "0x70"


def _driver_mistakes(cfg):
    cfg["devices"][0]["address"] = "0x44"
    cfg["devices"][1]["path"] = [["mux_a", 9]]
    cfg["devices"].append({"name": "x", "bus": 0, "address": 1, "driver": "NOPE"})


def _mux_at_default(cfg):
    cfg["muxes"]["mux"]["address"] = "0x70"


CONFIGS = {
    "small": json.dumps(SMALL),
    "bench": json.dumps(BENCH),
    "unknown mux and bus": variant(SMALL, _unknown_mux_and_bus),
    "duplicate name": variant(SMALL, _duplicate_name),
    "mux cycle": variant(SMALL, _cycle),
    "upstream collision": variant(SMALL, _upstream_collision),
    "driver mistakes": variant(SMALL, _driver_mistakes),
    "bench mux at 0x70": variant(BENCH, _mux_at_default),
}


HOST = {}  # filled from the host below: {config name: the host's sorted problems}


def board_problems(name):
    """Sorted problems the board finds in ``CONFIGS[name]``; ``[]`` if it is valid."""
    try:
        RoutingTable.loads(CONFIGS[name]).validate(REG)
    except ConfigError as err:
        return sorted(err.problems)
    return []


for name in CONFIGS:
    print("{:<22} {} problem(s)".format(name, len(board_problems(name))))

## Matching tests

**Matches** `test_valid_config_passes_and_parses_hex` and `test_json_text_accepted`: the
board's `json` and `int(text, 16)` give the same table.

**What to do:** nothing.

In [ ]:
table = RoutingTable.loads(CONFIGS["small"]).validate(REG)
check("test_valid_config_passes_and_parses_hex",
      table.muxes["mux_a"]["address"] == 0x71
      and table.resolve_path(table.devices[1]["path"]) == [[0x71, 3]])  # fmt: skip
check("test_json_text_accepted",
      RoutingTable.loads('{"buses": [0], "devices": []}').validate() is not None)  # fmt: skip

**Matches** `test_unknown_mux_and_undeclared_bus`, `test_duplicate_names` and
`test_mux_cycle_rejected`.

**What to do:** nothing.

In [ ]:
text = "\n".join(board_problems("unknown mux and bus"))
check("test_unknown_mux_and_undeclared_bus",
      "unknown mux 'nope'" in text and "not in declared buses" in text)  # fmt: skip
check("test_duplicate_names", "duplicate device name" in "\n".join(board_problems("duplicate name")))
check("test_mux_cycle_rejected", "mux 'mux_a'" in "\n".join(board_problems("mux cycle")))
check_raises("validate raises ConfigError", ConfigError,
             RoutingTable.loads(CONFIGS["mux cycle"]).validate)  # fmt: skip

**Matches** `test_collision_seen_through_upstream_segment` and `test_driver_aware_checks`,
with the real driver classes on the board: `{:#04x}` and `hex()` format the same as on the
host.

**What to do:** nothing.

In [ ]:
text = "\n".join(board_problems("upstream collision"))
check("test_collision_seen_through_upstream", "address collision on bus 0 at 0x70" in text, text)
text = "\n".join(board_problems("driver mistakes"))
check("test_driver_aware_checks",
      "not valid for SHTC3" in text and "channel 9 out of range 0..7" in text
      and "'NOPE' not in registry" in text)  # fmt: skip

**Matches** `test_bench_config_validates` and
`test_bench_mux_at_default_address_collides_with_shtc3`, with the real registry: this is
`test_bench_validates_with_the_real_drivers` on the board.

**What to do:** nothing.

In [ ]:
check("test_bench_config_validates", board_problems("bench") == [])
check("test_bench_mux_at_default_collides",
      "collision on bus 0 at 0x70" in "\n".join(board_problems("bench mux at 0x70")))  # fmt: skip

## Board and host agree

Notebook-only: the host validates the same JSON texts with its own `RoutingTable` and the
real registry, and sends its sorted problems to the board as `HOST`. The texts come from the
board, so both sides parse exactly the same bytes.

**What to do:** nothing.

In [ ]:
%%host
host_configs = get_ipython().run_line_magic("eval", "CONFIGS")
host_names = get_ipython().run_line_magic("eval", "DRIVER_NAMES")
host_reg = registry(host_names)


def host_problems(text):
    """Sorted problems the host finds in ``text``; ``[]`` if it is valid."""
    try:
        RoutingTable.loads(text).validate(host_reg)
    except ConfigError as err:
        return sorted(err.problems)
    return []


found = {name: host_problems(text) for name, text in host_configs.items()}
sent = get_ipython().run_line_magic("eval", "HOST.update({!r}) or len(HOST)".format(found))
print("sent the host's problems for {} configs".format(sent))

**Matches** every unit test above that validates a config: for each config the board and the
host must find exactly the same problems, word for word.

**What to do:** nothing. A `FAIL` line prints the board's problems for that config.

In [ ]:
for name in CONFIGS:
    mine = board_problems(name)
    check("same problems: " + name, mine == HOST.get(name), "" if mine == HOST.get(name) else repr(mine))

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))